# 개체별 상태 전환 방식 재검증

질문 → 전체 영상 후보 탐색(4초 창, 0.5초 프레임) → 후보 주변 검증(0.25초 프레임) → 전·후 프레임 독립 확인 → 동일 개체 연결과 행동 사이 리셋 확인 → 근거 시각을 남기는 실험입니다. 원본 영상은 로컬 `data/videos/`에 있고, 저장된 JSON만 읽으므로 **노트북 실행에는 API 호출이나 DB 쓰기가 없습니다.**

API 크레딧 충전 후 동일 개체 연결과 울타리 질문 해석을 재실행했습니다. 아래는 저장된 **로컬 실험 결과**이며 `/query` 서버나 worker 배포 결과가 아닙니다. 사람 기준 중 `0518·0642·0608·0142`는 프레임 잠정 검토이며, `0065`는 사용자에게서 사건별 시각을 받지 못했습니다.


## 요약

| 영상 | 사람 기준 | 자동 확정 | 미확정 후보 | 사건 시각 일치 | 판정 |
|---|---:|---:|---:|---:|---|
| `0518` 소파 | 잠정 2 | 2 | 0 | 2/2 | 검은·흰 강아지 각각 포착 |
| `0608` 수영장 | 잠정 2 | 2 | 1 | 2/2 | 두 입수 포착, 지속 수영 후보 미확정 |
| `0642` 분수 | 잠정 2 | 1 | 1 | 1/2 | 두 번째 검은 강아지 입수 미확정 |
| `0142` 고개 들기 | 잠정 2 | 2 | 1 | 2/2 | 같은 고양이의 연속 자세를 리셋 없이 중복 세지 않음 |
| `0936` 앞발 들기 | 사용자 8 | 3 | 2 | 3/8 | 빠른 행동 다수 누락 |
| `0065` 소파 | 사용자 확실 4 + 애매 1 | 0 | 2 | 측정 불가 | 사건별 사용자 시각 라벨 없음 |
| `0188` 울타리 | 사용자 2 (`dog_escape`와 같은 영상) | 2 | 3 | 2/2 | 약 4.5초·22.75초를 각 1회로 포착 |

`0065`의 자동 확정 `0`은 실제로 0회라는 뜻이 아닙니다. 모델이 근거를 충분히 내지 못해 확정하지 못했다는 뜻입니다. `0936`도 숫자와 사건 재현율이 아직 낮습니다.


In [ ]:
import json
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks": root = root.parent
evidence = root / "notebooks/smarthome-bench-evidence/new-pipeline"
summary = json.loads((evidence / "evaluation-summary.json").read_text(encoding="utf-8"))
for c in summary["cases"]:
    match = c["time_only_matches"]
    matched = len(match["matches"]) if match else "시각 라벨 없음"
    print(c["video_id"], "사람", c["reference_confirmed"],
          "자동 확정", c["pipeline_confirmed"], "미확정 후보", c["pipeline_uncertain_candidates"],
          "시각 일치", matched, "개체 연결", c["track_status"])


## 후보별 전·후 근거와 실패 원인

`confirmed`는 모델 검증을 통과한 후보, `uncertain`은 전환이나 개체·리셋 근거가 부족한 후보입니다. `rejected`는 모델이 목표 상태 진입을 확인하지 못한 후보입니다. `evidence_check`는 모델이 제시한 전·후 프레임을 따로 검사한 응답입니다. 이 검사도 완전하지 않아 `0142`에서 같은 고양이의 머리 자세를 잘못 판단했습니다.


In [ ]:
for case in summary["cases"]:
    video_id = case["video_id"]
    result = json.loads((evidence / f"{video_id}-result.json").read_text(encoding="utf-8"))
    print("\n###", video_id, result["query"])
    print("상태 전환:", result["intent"]["before_state"], "→", result["intent"]["after_state"])
    for item in result["verifications"]:
        row = item["observation"]
        check = item.get("repair_evidence_check") or item.get("evidence_check") or {}
        print(f'  후보 {item["candidate"]["time"]:>5.2f}s  {row.get("status", "unknown"):<9} '
              f'개체={row.get("subject_id")}  전={row.get("before_time")}  후={row.get("after_time")}'
              f'  근거 재검사={check.get("status", "없음")}')


## 원본 영상 확인

아래 셀은 로컬 Jupyter에서 영상을 재생합니다. `0518`은 약 19~22초, `0642`는 약 1~6초, `0142`는 약 10~28초, `0936`은 0~20초를 보세요. `0065`는 사용자 기준 4확실+1애매의 **각 시각**을 기록해야 사건 단위 성능을 평가할 수 있습니다.


In [ ]:
from IPython.display import Video, display
for case in summary["cases"]:
    path = root / "data/videos" / f'{case["video_id"]}.mp4'
    print(case["video_id"], path if path.exists() else "로컬 영상 없음")
    if path.exists(): display(Video(filename=str(path), embed=False, html_attributes="controls"))


## 다음 수정 우선순위

1. **빠른 행동 후보 재현율:** `0936`처럼 1초 미만의 동작은 현재 0.5초 탐색에서 빠집니다. 움직임 기반 후보 제안이나 더 촘촘한 탐색이 필요합니다.
2. **비슷한 개체 구분:** `0642`에서는 두 번째 검은 강아지의 입수 전 상태와 개체 ID가 충분히 확인되지 않았습니다. 자동으로 2회라고 단정하지 않고 미확정으로 둡니다.
3. **소파 사건 시각 라벨:** `0065`는 사용자가 센 4회 확실+1회 애매의 개별 시각을 기록해야 사건 단위 성능을 비교할 수 있습니다.
4. **근거 재검사 신뢰도:** 전·후 프레임과 행동 사이 리셋을 별도로 검사해 `0142`와 `0188`은 개선됐지만, 한 프레임의 자세를 모델이 잘못 읽을 가능성은 남아 있습니다.

현재 결과는 발표에서 **방법, 맞은 사건, 남은 실패 사례를 함께 설명할 수 있는 로컬 검증 자료**입니다. 모든 행동의 자동 횟수 세기가 해결됐다고 주장할 근거는 아직 없습니다.
